In [7]:
# import libraries
import requests
import pandas as pd
import numpy as np
from time import sleep
import json

In [8]:
# read in data
ncbi = pd.read_csv('../data/raw/sequences.csv', low_memory=False)

# Quick overview
print(ncbi.shape)
print(ncbi.columns.tolist())
ncbi.head()

(3256603, 14)
['Accession', 'Organism_Name', 'Species', 'Genotype', 'Isolate', 'Length', 'Nuc_Completeness', 'Geo_Location', 'Host', 'Tissue_Specimen_Source', 'Submitters', 'Collection_Date', 'Release_Date', 'Molecule_type']


,Accession,Organism_Name,Species,Genotype,Isolate,Length,Nuc_Completeness,Geo_Location,Host,Tissue_Specimen_Source,Submitters,Collection_Date,Release_Date,Molecule_type
0,NC_138438,Red panda feces-associated circular DNA virus 12,Alohovirus ailgensis,NaN,Rpf011unssDNA01-5,2820,complete,China: Sichuan Province,Ailurus fulgens,feces,"Zhao,M., Yue,C., Yang,Z., Li,Y., Zhang,D., Zha...",2020-05,2026-04-07,ssDNA
1,NC_138461,Red panda feces-associated circular DNA virus 11,Protegovirus ailuris,NaN,AliP02cress04-2015,2497,complete,China: Sichuan Province,Ailurus fulgens,feces,"Zhao,M., Yue,C., Yang,Z., Li,Y., Zhang,D., Zha...",2015,2026-04-07,ssDNA
2,NC_097195,Pacific flying fox faeces associated circular ...,Pekavirus fuais,NaN,Tbat_38855,2214,complete,Tonga,Pteropus tonganus,feces,"Male,M.F., Kraberger,S., Stainton,D., Kami,V.,...",2014,2026-04-01,ssDNA
3,NC_099061,Bat circovirus POA/2012/V,Vegetinivirus molmolis,NaN,cluster V,1728,complete,Brazil,Chiroptera,feces,"Lima,F.E., Cibulski,S.P., Dos Santos,H.F., Tei...",2012-02-13,2026-04-01,ssDNA
4,NC_116632,Delphin virus 1,Baranavirus uduis,NaN,3_2016_1939,1939,complete,Saint Vincent and the Grenadines,Orcinus orca,muscle,"Smith,K., Fielding,R., Schiavone,K., Hall,K., ...",2015-08-26,2026-04-01,ssDNA


In [5]:
class NCBIPhyloEstimator:
    """
    Estimates phylogenetic divergence times from Homo sapiens.

    Lookup priority:
      1. Direct match against expanded table (orders, families, common genera)
      2. Genus extraction from binomial name -> table lookup
      3. NCBI Taxonomy API fallback with retry/backoff for unknowns

    Divergence times (MYA) from TimeTree consensus (Kumar et al. 2017).
    Steps 1-2 cover the vast majority of named vertebrate hosts with no API calls.
    """

    DIVERGENCE_TABLE = {
        # Primates – genus level
        'Pan': 6.9, 'Gorilla': 9.1, 'Pongo': 15.2,
        'Nomascus': 20.0, 'Symphalangus': 20.0, 'Hylobates': 20.0,
        # Cercopithecidae
        'Macaca': 29.0, 'Papio': 29.0, 'Cercocebus': 29.0, 'Mandrillus': 29.0,
        'Theropithecus': 29.0, 'Chlorocebus': 29.0, 'Erythrocebus': 29.0,
        'Cercopithecus': 29.0, 'Colobus': 29.0, 'Procolobus': 29.0,
        'Presbytis': 29.0, 'Trachypithecus': 29.0, 'Semnopithecus': 29.0,
        'Nasalis': 29.0, 'Cercopithecidae': 29.0,
        # New World monkeys
        'Aotus': 43.0, 'Saimiri': 43.0, 'Callithrix': 43.0, 'Cebus': 43.0,
        'Ateles': 43.0, 'Alouatta': 43.0, 'Platyrrhini': 43.0,
        # Strepsirrhini
        'Lemur': 74.0, 'Microcebus': 74.0, 'Eulemur': 74.0,
        'Loris': 74.0, 'Nycticebus': 74.0, 'Galago': 74.0,
        'Strepsirrhini': 74.0, 'Primates': 74.0,
        # Treeshrews / Colugos
        'Tupaia': 82.0, 'Scandentia': 82.0, 'Dermoptera': 82.0,
        # Rodentia – common genera
        'Mus': 87.0, 'Rattus': 87.0, 'Apodemus': 87.0, 'Mastomys': 87.0,
        'Grammomys': 87.0, 'Micaelamys': 87.0, 'Praomys': 87.0,
        'Peromyscus': 87.0, 'Sigmodon': 87.0, 'Reithrodontomys': 87.0,
        'Neotoma': 87.0, 'Oryzomys': 87.0, 'Oligoryzomys': 87.0,
        'Microtus': 87.0, 'Clethrionomys': 87.0, 'Myodes': 87.0,
        'Marmota': 87.0, 'Spermophilus': 87.0, 'Dipodomys': 87.0,
        'Chaetodipus': 87.0, 'Hydrochoerus': 87.0, 'Cavia': 87.0,
        'Cricetulus': 87.0, 'Mesocricetus': 87.0, 'Phodopus': 87.0,
        'Rodentia': 87.0,
        # Lagomorpha
        'Lepus': 87.0, 'Oryctolagus': 87.0, 'Lagomorpha': 87.0,
        # Chiroptera – common genera
        'Rhinolophus': 95.0, 'Hipposideros': 95.0, 'Pteropus': 95.0,
        'Rousettus': 95.0, 'Eidolon': 95.0, 'Epomophorus': 95.0,
        'Tadarida': 95.0, 'Molossus': 95.0, 'Mops': 95.0, 'Chaerephon': 95.0,
        'Myotis': 95.0, 'Eptesicus': 95.0, 'Pipistrellus': 95.0,
        'Nyctalus': 95.0, 'Vespertilio': 95.0, 'Cnephaeus': 95.0,
        'Miniopterus': 95.0, 'Murina': 95.0, 'Tylonycteris': 95.0,
        'Plecotus': 95.0, 'Desmodus': 95.0, 'Artibeus': 95.0,
        'Uroderma': 95.0, 'Sturnira': 95.0, 'Carollia': 95.0,
        'Hipposideridae': 95.0, 'Rhinolophidae': 95.0, 'Pteropodidae': 95.0,
        'Vespertilionidae': 95.0, 'Molossidae': 95.0, 'Phyllostomidae': 95.0,
        'Chiroptera': 95.0,
        # Eulipotyphla
        'Sorex': 94.0, 'Blarina': 94.0, 'Crocidura': 94.0,
        'Erinaceus': 94.0, 'Talpa': 94.0, 'Eulipotyphla': 94.0,
        # Carnivora
        'Felis': 96.0, 'Panthera': 96.0, 'Lynx': 96.0, 'Puma': 96.0,
        'Canis': 96.0, 'Vulpes': 96.0, 'Nyctereutes': 96.0,
        'Neogale': 96.0, 'Mustela': 96.0, 'Neovison': 96.0,
        'Ursus': 96.0, 'Ailuropoda': 96.0, 'Ailurus': 96.0,
        'Phoca': 96.0, 'Halichoerus': 96.0, 'Leptonychotes': 96.0,
        'Mirounga': 96.0, 'Hyaena': 96.0, 'Crocuta': 96.0,
        'Carnivora': 96.0,
        # Cetartiodactyla
        'Sus': 95.0, 'Bos': 95.0, 'Ovis': 95.0, 'Capra': 95.0,
        'Cervus': 95.0, 'Alces': 95.0, 'Rangifer': 95.0, 'Odocoileus': 95.0,
        'Camelus': 95.0, 'Vicugna': 95.0, 'Lama': 95.0,
        'Orcinus': 95.0, 'Tursiops': 95.0, 'Delphinus': 95.0,
        'Balaenoptera': 95.0, 'Cetacea': 95.0,
        'Artiodactyla': 95.0, 'Cetartiodactyla': 95.0,
        # Perissodactyla
        'Equus': 95.0, 'Rhinoceros': 95.0, 'Tapirus': 95.0, 'Perissodactyla': 95.0,
        # Pholidota (pangolins)
        'Manis': 80.0, 'Smutsia': 80.0, 'Phataginus': 80.0, 'Pholidota': 80.0,
        # Afrotheria
        'Loxodonta': 95.0, 'Elephas': 95.0, 'Proboscidea': 95.0,
        'Procavia': 95.0, 'Orycteropus': 95.0,
        # Xenarthra
        'Choloepus': 95.0, 'Bradypus': 95.0, 'Dasypus': 95.0, 'Xenarthra': 95.0,
        # Marsupials / Monotremes
        'Macropus': 175.0, 'Monodelphis': 175.0, 'Didelphis': 175.0,
        'Marsupialia': 175.0, 'Ornithorhynchus': 177.0, 'Monotremata': 177.0,
        # Birds
        'Gallus': 312.0, 'Anas': 312.0, 'Anser': 312.0, 'Columba': 312.0,
        'Corvus': 312.0, 'Aves': 312.0,
        # Reptiles / Amphibians / Fish
        'Crocodylus': 312.0, 'Python': 312.0, 'Reptilia': 312.0,
        'Rana': 350.0, 'Xenopus': 350.0, 'Amphibia': 350.0,
        'Danio': 435.0, 'Oncorhynchus': 435.0, 'Actinopterygii': 435.0,
        'Chondrichthyes': 450.0, 'Vertebrata': 500.0,
        # Invertebrates
        'Drosophila': 700.0, 'Aedes': 700.0, 'Anopheles': 700.0,
        'Culex': 700.0, 'Ixodes': 700.0, 'Insecta': 700.0,
        'Arthropoda': 700.0, 'Nematoda': 800.0,
    }

    NCBI_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

    def __init__(self, ncbi_api_key=None):
        self.session = requests.Session()
        self._cache = {}
        self.ncbi_api_key = ncbi_api_key
        # 1s between requests without API key; 0.12s with key (10 req/sec limit)
        self._interval = 0.12 if ncbi_api_key else 1.0

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _lookup_direct(self, name):
        return self.DIVERGENCE_TABLE.get(name)

    def _lookup_genus(self, name):
        """Extract genus from a binomial/trinomial name and look it up."""
        parts = name.split()
        if len(parts) >= 2:
            return self.DIVERGENCE_TABLE.get(parts[0])
        return None

    def _ncbi_lookup(self, species_name):
        """Query NCBI Taxonomy with exponential backoff on 429."""
        if species_name in self._cache:
            return self._cache[species_name]

        def _params(base):
            p = dict(base)
            if self.ncbi_api_key:
                p['api_key'] = self.ncbi_api_key
            return p

        for attempt in range(4):
            try:
                r = self.session.get(
                    f"{self.NCBI_BASE}/esearch.fcgi",
                    params=_params({'db': 'taxonomy', 'term': species_name, 'retmode': 'json'}),
                    timeout=15
                )
                r.raise_for_status()
                ids = r.json().get('esearchresult', {}).get('idlist', [])
                if not ids:
                    self._cache[species_name] = None
                    return None

                sleep(self._interval)

                r2 = self.session.get(
                    f"{self.NCBI_BASE}/efetch.fcgi",
                    params=_params({'db': 'taxonomy', 'id': ids[0], 'retmode': 'json'}),
                    timeout=15
                )
                r2.raise_for_status()
                taxa = r2.json().get('result', {}).get(ids[0])

                # NCBI sometimes returns the taxid as a bare int for merged/redirected
                # entries instead of the full record dict – guard against that
                if not isinstance(taxa, dict):
                    self._cache[species_name] = None
                    return None

                lineage = [t.strip() for t in taxa.get('lineage', '').split(';') if t.strip()]
                sleep(self._interval)

                for taxon in reversed(lineage):
                    if taxon in self.DIVERGENCE_TABLE:
                        mya = self.DIVERGENCE_TABLE[taxon]
                        self._cache[species_name] = mya
                        return mya

                self._cache[species_name] = None
                return None

            except requests.exceptions.HTTPError as e:
                if e.response.status_code == 429:
                    wait = 2 ** (attempt + 1)   # 2, 4, 8, 16 s
                    print(f"  Rate limited, retrying in {wait}s...")
                    sleep(wait)
                    continue
                print(f"NCBI HTTP error for '{species_name}': {e}")
                break
            except Exception as e:
                print(f"NCBI lookup failed for '{species_name}': {e}")
                break

        self._cache[species_name] = None
        return None

    # ------------------------------------------------------------------
    # Public interface (same as original TimeTreeExtractor)
    # ------------------------------------------------------------------

    def get_divergence_time(self, species1, species2="Homo sapiens"):
        """
        Estimate divergence time (MYA) between species1 and Homo sapiens.
        species2 is accepted for API compatibility; only human-referenced
        distances are supported.
        """
        if species1.lower() in ('homo sapiens', 'human', 'humans'):
            return 0.0

        # 1. Exact match (handles order/family/genus-only host entries)
        result = self._lookup_direct(species1)
        if result is not None:
            return result

        # 2. Genus from binomial – covers named species with no API call
        result = self._lookup_genus(species1)
        if result is not None:
            return result

        # 3. NCBI fallback for anything not in the table
        return self._ncbi_lookup(species1)

    def _estimate_distance(self, species1, species2):
        return np.nan

    def calculate_phylogenetic_matrix(self, species_list):
        """
        Pairwise divergence matrix. Distance approximated as the divergence
        time of the earlier-diverging species from humans.
        """
        n = len(species_list)
        matrix = np.zeros((n, n))
        human_dists = {sp: self.get_divergence_time(sp) for sp in species_list}
        print(f"Calculating distances for {n} species...")

        for i, sp1 in enumerate(species_list):
            for j, sp2 in enumerate(species_list):
                if i == j:
                    matrix[i, j] = 0
                elif i < j:
                    d1 = human_dists.get(sp1)
                    d2 = human_dists.get(sp2)
                    dist = max(d1, d2) if (d1 is not None and d2 is not None) else np.nan
                    matrix[i, j] = dist
                    matrix[j, i] = dist

                if (i * n + j + 1) % 10 == 0:
                    progress = ((i * n + j + 1) / (n ** 2)) * 100
                    print(f"Progress: {progress:.1f}%")

        return pd.DataFrame(matrix, index=species_list, columns=species_list)

In [6]:
def calculate_human_distances(host_species_list):
    """
    Calculate phylogenetic distances from each host to humans.

    Args:
        host_species_list (list): List of host species scientific names

    Returns:
        pandas.DataFrame: Host species with human distances and risk scores
    """
    estimator = NCBIPhyloEstimator()
    results = []

    for host in host_species_list:
        print(f"Processing {host}...")
        divergence_mya = estimator.get_divergence_time(host)

        if divergence_mya is not None:
            phylo_risk_score = np.exp(-divergence_mya / 50)  # 50 MYA half-life

            if divergence_mya < 20:
                risk_category = "Very High"
            elif divergence_mya < 50:
                risk_category = "High"
            elif divergence_mya < 100:
                risk_category = "Medium"
            else:
                risk_category = "Low"

            results.append({
                'host_species': host,
                'human_divergence_mya': divergence_mya,
                'phylogenetic_risk_score': phylo_risk_score,
                'risk_category': risk_category
            })
        else:
            results.append({
                'host_species': host,
                'human_divergence_mya': None,
                'phylogenetic_risk_score': 0.1,
                'risk_category': 'Unknown'
            })

    return pd.DataFrame(results)


host_species = [
    "Rhinolophus sinicus",  # Chinese horseshoe bat
    "Manis javanica",       # Pangolin
    "Sus scrofa",           # Wild boar
    "Felis catus",          # Domestic cat
    "Rattus norvegicus",    # Norway rat
    "Eptesicus fuscus"      # Big brown bat
]

human_distance_df = calculate_human_distances(host_species)
print(human_distance_df)

Processing Rhinolophus sinicus...
Processing Manis javanica...
Processing Sus scrofa...
Processing Felis catus...
Processing Rattus norvegicus...
Processing Eptesicus fuscus...
          host_species  human_divergence_mya  phylogenetic_risk_score  \
0  Rhinolophus sinicus                  95.0                 0.149569   
1       Manis javanica                  80.0                 0.201897   
2           Sus scrofa                  95.0                 0.149569   
3          Felis catus                  96.0                 0.146607   
4    Rattus norvegicus                  87.0                 0.175520   
5     Eptesicus fuscus                  95.0                 0.149569   

  risk_category  
0        Medium  
1        Medium  
2        Medium  
3        Medium  
4        Medium  
5        Medium  


In [ ]:
# Get unique hosts from dataset
host_species_list = ncbi['Host'].dropna().unique().tolist()

# Calculate phylogenetic risk vs humans for all hosts
human_distance_df = calculate_human_distances(host_species_list)

# Merge back into main dataframe
ncbi = ncbi.merge(human_distance_df, left_on='Host', right_on='host_species', how='left').drop(columns='host_species')

print(ncbi[['Host', 'human_divergence_mya', 'phylogenetic_risk_score', 'risk_category']].drop_duplicates().sort_values('phylogenetic_risk_score', ascending=False).head(20))